<a href="https://colab.research.google.com/github/kokilahemachandra-afk/srilanka-ntl-midterm/blob/main/notebooks/02_build_panel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Clean the monthly lights and build the panels

**Input:** `data/processed/lka_ds339_ntl_monthly.csv` (from notebook 01)
and `data/boundaries/ds_units.csv` (names, districts, provinces).

**Output:**
- `lka_ds339_ntl_monthly_clean.csv`: monthly data with a quality flag
- `lka_ds339_ntl_annual.csv`: one row per DS division and year (2014–2025),
  used for the spatial analysis in notebook 03

##Step 4 · Clean the monthly data and build the yearly panel

### Step 4.1 · Read the data from GitHub
Both files are read directly from this repository, so the notebook runs anywhere.

In [3]:
import pandas as pd
import plotly.express as px

REPO = "https://raw.githubusercontent.com/kokilahemachandra-afk/srilanka-ntl-midterm/main/"

ntl   = pd.read_csv(REPO + "data/processed/lka_ds339_ntl_monthly.csv")
import geopandas as gpd

shp = gpd.read_file(REPO + "data/boundaries/lka_admin3.shp")  # your adm3 file name
units = (shp.rename(columns={"adm3_pcode": "geo_id", "adm3_name": "name",
                             "adm2_name": "district", "adm2_pcode": "district_id",
                             "adm1_name": "province", "adm1_pcode": "province_id"})
            [["geo_id", "name", "district", "district_id", "province", "province_id", "area_sqkm"]])
print(len(units), "DS divisions")

units.to_csv("ds_units.csv", index=False)    # save a copy so you can upload it

ntl = ntl.merge(units, on="geo_id")                       # add name, district, province
ntl["date"] = pd.to_datetime(dict(year=ntl["year"], month=ntl["month"], day=1))

print(ntl.shape, "|", ntl["geo_id"].nunique(), "DS divisions |",
      ntl["date"].min().date(), "to", ntl["date"].max().date())
ntl.head()

339 DS divisions
(51867, 13) | 339 DS divisions | 2014-01-01 to 2026-09-01


,geo_id,year,month,avg_rad_sum,avg_rad_mean,cf_cvg_mean,name,district,district_id,province,province_id,area_sqkm,date
0,LK1103,2014,1,3767.721402,32.787392,6.488653,Colombo,Colombo,LK11,Western,LK1,24.537151,2014-01-01
1,LK1106,2014,1,959.140454,7.878775,6.405341,Kolonnawa,Colombo,LK11,Western,LK1,25.928037,2014-01-01
2,LK1109,2014,1,1580.817712,3.872804,6.636986,Kaduwela,Colombo,LK11,Western,LK1,86.737826,2014-01-01
3,LK1112,2014,1,1029.980301,1.818481,7.124405,Homagama,Colombo,LK11,Western,LK1,120.316227,2014-01-01
4,LK1115,2014,1,479.703745,0.698326,6.753557,Seethawaka,Colombo,LK11,Western,LK1,145.976978,2014-01-01


### Step 4.2 · Flag cloudy months
When a division has fewer than one cloud-free night in a month, the satellite
barely saw it and the light value is unreliable (often zero). This happens
mostly in the monsoon months. We flag these division-months and leave them out
of the averages.

In [4]:
ntl["low_quality"] = ntl["cf_cvg_mean"] < 1

print("Flagged:", ntl["low_quality"].sum(), "division-months",
      f"({ntl['low_quality'].mean():.1%} of all)")
print("Zero values left after removing flagged months:",
      ((ntl["avg_rad_sum"] == 0) & ~ntl["low_quality"]).sum())

# Share of flagged division-months by calendar month
ntl.groupby("month")["low_quality"].mean().round(3)

Flagged: 2223 division-months (4.3% of all)
Zero values left after removing flagged months: 0


,low_quality
month,
1,0.002
2,0.000
3,0.000
4,0.000
5,0.044
6,0.037
7,0.161
8,0.099
9,0.068


### Step 4.3 · The national picture, month by month
Total light of all 339 divisions in each month. The red points are months where
more than 20% of divisions were flagged as cloudy: those dips are clouds, not
real falls in activity.

In [5]:
nat = ntl.groupby("date", as_index=False).agg(
    total_light=("avg_rad_sum", "sum"),
    share_cloudy=("low_quality", "mean"))
nat["cloudy_month"] = nat["share_cloudy"] > 0.2

fig = px.line(nat, x="date", y="total_light",
              title="Sri Lanka: total night-time light per month (339 DS divisions)",
              labels={"total_light": "Total radiance", "date": ""})
cloudy = nat[nat["cloudy_month"]]
fig.add_scatter(x=cloudy["date"], y=cloudy["total_light"], mode="markers",
                marker=dict(color="red", size=8), name="> 20% of divisions cloudy")
fig.show()

### Step 4.4 · Build the yearly panel
For each division and year we average the **good months only**. Averaging
removes most of the month-to-month noise, and the spatial methods (Moran's I,
LISA, convergence) work on yearly values.
We use the complete years 2014–2025 (2026 is not finished yet).
`months` records how many good months each average is based on.

In [6]:
good = ntl[~ntl["low_quality"] & (ntl["year"] <= 2025)]

annual = (good.groupby(["geo_id", "year"], as_index=False)
              .agg(ntl_sum=("avg_rad_sum", "mean"),
                   ntl_mean=("avg_rad_mean", "mean"),
                   months=("month", "count")))
annual = annual.merge(units, on="geo_id")

print(annual.shape, "| years:", annual["year"].min(), "to", annual["year"].max())
print("Good months per division-year: min", annual["months"].min(),
      "| average", round(annual["months"].mean(), 1))
annual.head()

(4068, 11) | years: 2014 to 2025
Good months per division-year: min 9 | average 11.5


,geo_id,year,ntl_sum,ntl_mean,months,name,district,district_id,province,province_id,area_sqkm
0,LK1103,2014,3449.895688,30.021616,12,Colombo,Colombo,LK11,Western,LK1,24.537151
1,LK1103,2015,3411.114049,29.684131,12,Colombo,Colombo,LK11,Western,LK1,24.537151
2,LK1103,2016,3611.745159,31.430059,11,Colombo,Colombo,LK11,Western,LK1,24.537151
3,LK1103,2017,3800.753181,33.074841,12,Colombo,Colombo,LK11,Western,LK1,24.537151
4,LK1103,2018,4018.287103,34.967860,12,Colombo,Colombo,LK11,Western,LK1,24.537151


### Step 4.5 · Growth by province
Total yearly light per province (sum of its divisions). The vertical axis is on
a log scale, so equal slopes mean equal growth rates.

In [10]:
prov = annual.groupby(["province", "year"], as_index=False)["ntl_sum"].sum()
fig = px.line(prov, x="year", y="ntl_sum", color="province", log_y=True, markers=True,
              title="Night-time light by province, 2014–2025",
              labels={"ntl_sum": "Total radiance (log scale)", "year": ""})
fig.update_xaxes(dtick=1, tickformat="d", fixedrange=True)   # one tick per year, no zoom
fig.update_yaxes(fixedrange=True)
fig.show()

### Step 4.6 · Save
Download both files and upload them to `data/processed/` in GitHub.

In [8]:
ntl.drop(columns="date").to_csv("lka_ds339_ntl_monthly_clean.csv", index=False)
annual.to_csv("lka_ds339_ntl_annual.csv", index=False)

from google.colab import files
files.download("lka_ds339_ntl_monthly_clean.csv")
files.download("lka_ds339_ntl_annual.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>